<p align="center">
  <img src="logo_habitca.webp" width="300" alt="Diagrama de clases"><br>
</p>

# ¿Qué es Habitica?
Habitica es una aplicación que convierte las tareas en un juego de rol: el usuario es un personaje que gana experiencia y oro al cumplir sus metas y pierde vida cuando las descuida. Este proyecto simula ese escenario en Python, aplicando programación orientada a objetos, además lo extiende con elementos propios que consisten en visualizar la constancia del usuario en un *calendario* y refuerza la motivación mediante la cooperación en *clanes*.

El sistema gira en torno al Jugador, que tiene nivel, experiencia, vida, oro e inventario, y organiza su día con tres tipos de tareas: los hábitos, las tareas diarias (con racha de cumplimiento) y las tareas pendientes (con fecha límite). Cumplir una tarea otorga recompensas según su dificultad, y fallar una diaria resta vida. Además, el jugador puede adoptar una clase de personaje (Aprendiz, Guerrero, Mago, Sanador o Pícaro), cada una con una ventaja distinta. Con el oro ganado, accede a una tienda de equipo, pociones y recompensas personales.

## ClasePersonaje
ClasePersonaje es la clase base de los roles del jugador. 

Las clases hijas almacenan las bonificaciones según el rol.

### *Atributos*
| Atributo | Descripción | Valor por defecto |
|---|---|---|
| `nombre` | Nombre de la clase | – |
| `descripcion` | Breve descripción de la clase | – |
| `mult_exp` | Multiplicador de experiencia | 1.0 |
| `mult_oro` | Multiplicador de oro | 1.0 |
| `reduccion_dano` | Fracción del daño que se evita | 0.0 |
| `curacion_nivel` | Fracción de vida que se restaura al subir de nivel | 0.5 |

### *Métodos*
| Método | Qué hace |
|---|---|
| `modificar_exp(base)` | Devuelve la experiencia con el multiplicador aplicado |
| `modificar_oro(base)` | Devuelve el oro con el multiplicador aplicado |
| `modificar_dano(base)` | Devuelve el daño final (mínimo 1 si hubo daño) |
| `curacion_al_subir_nivel(hp_maxima)` | Devuelve la vida que se restaura al subir de nivel |
| `__str__()` | Devuelve "Nombre: descripción" |

### *Clases hijas*
| Clase | Descripción | Modificador |
|---|---|---|
| Aprendiz | Clase inicial, sin bonificaciones | Ninguno |
| Guerrero | Muy resistente contra el daño | reduccion_dano = 0.30 |
| Mago | Sube de nivel más rápido | mult_exp = 1.25 |
| Sanador | Restaura su salud al subir de nivel | curacion_nivel = 1.0 |
| Picaro | Obtiene oro más rápido | mult_oro = 1.25 |


## Tarea
Tarea es la clase base para cada tipo de Tarea que puede realizar el Jugador.

Las clases hijas modifican recompensas, formas de almacenar el historial y formas de completación según el tipo.

### *Atributos*
| Atributo | Descripción | Valor por defecto |
|---|---|---|
| `titulo` | Nombre de la Tarea | – |
| `dificultad` | Nivel de dificultad de la Tarea | - |
| `descripcion` | Detalles adicionales sobre la Tarea | None |
| `historial` | Lista de fechas en las que la Tarea fue completada | [] |
| `fecha_creacion` | Fecha en la que se creo un objeto de la clase Tarea | Fecha actual del sistema |
| `RECOMPENSAS` | Fracción de vida que se restaura al subir de nivel | Diccionario con las recompensas según la dificultad |

### *Métodos*
| Método | Qué hace |
|---|---|
| `editar_tarea(titulo, dificultad, descripcion)` | Modifica el atributo que se especifíque |
| `calcular_recompensa()` | Devuelve una recompensa según la dificultad de la Tarea |
| `completar()` | Agrega la fecha actual del sistema al historial y devuelve una recompensa |
| `__repr__()` | Devuelve informacion sobre los atributos de la Tarea |

### *Clases hijas*
| Clase | Descripción | Atributos propios |
|---|---|---|
| Habito | Representa un Habito positivo o negativo de un Jugador | TIPOS_HABITO, tipo_habito = "Ambos", historial_positivo = [], historial_negativo = [], contador_positivo = 0, contador_negativo = 0 |
| Diaria | Representa una Tarea que el Jugador debe completar una vez al día | racha = 0, mejor_racha = 0, fecha_inicio_racha = None |
| Pendiente | Representa una Tarea con fecha límite de un Jugador | fecha_limite, completada = False, vencida = False |


## Calendario
Esta clase administra las fechas en que se completo una Tarea. 

Tiene el objetivo de mostrar un calendario.

### *Atributos*
| Atributo | Descripción | Valor por defecto |
|---|---|---|
| `tarea` | Tarea de la cual se busca mostrar su calendario | – |

### *Métodos*
| Método | Qué hace |
|---|---|
| `mostrar` | Muestra un calendario de máximo tres meses coloreado con las fechas en las que se completo una Tarea. |


## Objetos de Tienda
Esta sección maneja la lógica de las compras en el juego. Se compone de una jerarquía de herencia para los ítems (`ItemTienda` y sus hijas) y una clase administradora (`Tienda`).

### 1. ItemTienda (Clase Base) y Clases Hijas
`ItemTienda` es la clase base para cualquier objeto que se pueda comprar en el juego.

#### *Atributos*
| Atributo | Descripción | Valor por defecto |
|---|---|---|
| `_contador_codigo` | Atributo de clase autoincremental para generar códigos únicos | 1 |
| `codigo` | Código único del ítem asignado al crearse | Generado automáticamente |
| `nombre` | Nombre del ítem | - |
| `precio` | Costo del ítem en oro | - |
| `nivel_requerido` | Nivel mínimo que debe tener el jugador para comprarlo | - |
| `descripcion` | Breve descripción del ítem | - |

#### *Métodos*
| Método | Qué hace |
|---|---|
| `aplicar(jugador)` | Aplica el efecto del ítem al jugador. En la clase base lanza `NotImplementedError`. |

#### *Clases Hijas*
| Clase | Atributos propios | Comportamiento en `aplicar()` |
|---|---|---|
| **Equipo** | `bonificacion` (int) | Se guarda en el inventario y suma la bonificación a la vida máxima del jugador. Retorna `False` si ya está equipado. |
| **Pocion** | `hp_restaurada` (int) | Usa el método `curar()` del jugador respetando su vida máxima. Retorna `False` si la vida ya está llena. |
| **Recompensa** | `creada_por_usuario` (bool)| Imprime un mensaje de canje exitoso para premios personales. |

---

### 2. Tienda
Esta clase administra la lista de objetos disponibles y maneja las transacciones.

#### *Atributos*
| Atributo | Descripción | Valor por defecto |
|---|---|---|
| `items` | Lista que almacena los objetos instanciados de tipo `ItemTienda` | `[]` (Lista vacía) |

#### *Métodos*
| Método | Qué hace |
|---|---|
| `agregar_item(item)` | Añade un objeto `ItemTienda` a la lista `items` tras validar que sea del tipo correcto y no esté duplicado. |
| `mostrar_tabla()` | Imprime en consola todos los ítems disponibles con su código, nombre, nivel, precio y descripción. |
| `buscar_por_codigo(codigo)` | Devuelve el objeto `ItemTienda` que coincide con el código dado, o `None` si no existe. |
| `comprar(jugador, codigo)` | Valida que el ítem exista, que el jugador tenga el nivel y oro suficientes. Si todo es correcto, ejecuta el método `aplicar()` del ítem y le descuenta el oro al jugador. |